# Day 36: Freshness Check Demo

This notebook demonstrates how to validate **data freshness** using the data quality framework.

## Overview
Freshness checks ensure that data is recent enough for analysis by verifying that the most recent record in a table is within an acceptable time window.

## What This Demo Covers
* Loading test data with timestamps
* Checking if data is fresh (within 24-hour threshold)
* Detecting stale data pipelines
* Using the `check_freshness()` and `check_all_freshness()` functions
* Saving results to a Delta table for tracking and reporting

## Test Scenario
* **Test Data**: Orders table with order_date timestamps
* **Fresh Data**: Orders from today (within 24 hours)
* **Stale Data**: Orders older than 24 hours
* **Expected Issue**: Some tables will have stale data that fails the freshness check

## Key Functions
* `check_freshness(df, timestamp_column, max_age_hours=24)` - Validates data freshness
* `check_all_freshness(table_registry, freshness_config)` - Batch freshness checks across multiple tables

## Output
* Freshness validation report showing latest timestamp, age in hours, and pass/fail status
* Results saved to `workspace.default.freshness_report` table

## 📝 Design Note: Why This Notebook Won't Fail Over Time

### The Problem
If we hardcode timestamps like `datetime(2026, 9, 16, 10, 0, 0)`, the test would work today but **fail tomorrow** because those fixed dates become stale.

### The Solution: Dynamic Timestamps
This notebook uses **relative timestamps** that are calculated fresh every time you run it:

```python
now = datetime.now()  # Gets current time on EVERY run

# Fresh data: always 2-20 hours ago from NOW
fresh_data = [
    (1, 101, 500, now - timedelta(hours=2)),   # Always 2 hours old
    (2, 102, 700, now - timedelta(hours=5)),   # Always 5 hours old
    ...
]

# Stale data: always 48+ hours ago from NOW
stale_data = [
    (5, 105, 600, now - timedelta(hours=48)),  # Always 48 hours old
    ...
]
```

### Why This Matters

| Run Date | `datetime.now()` | Fresh Data Age | Stale Data Age | Result |
|----------|------------------|----------------|----------------|--------|
| **Today** (Sep 16) | Sep 16, 1:00 PM | 2-20 hours | 48+ hours | ✅ Fresh PASS, ❌ Stale FAIL |
| **Tomorrow** (Sep 17) | Sep 17, 1:00 PM | 2-20 hours | 48+ hours | ✅ Fresh PASS, ❌ Stale FAIL |
| **Next Month** (Oct 20) | Oct 20, 1:00 PM | 2-20 hours | 48+ hours | ✅ Fresh PASS, ❌ Stale FAIL |

### Key Takeaway
✅ **This notebook will produce consistent results every time you run it** — today, tomorrow, next year.

✅ **Best Practice:** Real-world data quality monitoring checks freshness **relative to "now"**, not against a fixed date.

✅ **Portfolio Value:** Shows you understand test repeatability and production-ready design patterns.

In [0]:
"""
Freshness Check Demo

This script demonstrates how to validate data freshness using timestamp columns.
It checks that the most recent record in a table is within an acceptable age threshold.

Freshness Validation:
    Check if max(order_date) is within 24 hours of current time
    
Expected Results:
    - Fresh tables will pass (data within 24 hours)
    - Stale tables will fail (data older than 24 hours)
"""

# ============================================================================
# 1. SETUP: Import required libraries
# ============================================================================
import sys
import yaml
import os
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType

# ============================================================================
# 2. DYNAMIC PATH CONFIGURATION
# ============================================================================
# Get the base repository path dynamically to avoid hard-coded paths
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# Go up three levels: day36 -> phase2_dq_framework -> notebooks -> data-quality-testing (base)
base_path = os.path.dirname(os.path.dirname(os.path.dirname(notebook_path)))

# ============================================================================
# 3. IMPORT DQ CHECK FUNCTIONS
# ============================================================================
# Add DQ checks module to Python path
checks_path = os.path.join("/Workspace", base_path.lstrip("/"), "src/checks")
sys.path.append(checks_path)
from dq_checks import check_freshness, check_all_freshness

# ============================================================================
# 4. CREATE SAMPLE DATA
# ============================================================================
# Create fresh data (within 24 hours)
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("amount", IntegerType(), True),
    StructField("order_date", TimestampType(), True)
])

now = datetime.now()

# Fresh orders (within 24 hours)
fresh_data = [
    (1, 101, 500, now - timedelta(hours=2)),
    (2, 102, 700, now - timedelta(hours=5)),
    (3, 103, 300, now - timedelta(hours=12)),
    (4, 104, 400, now - timedelta(hours=20))
]
orders_fresh = spark.createDataFrame(fresh_data, schema)

print("📊 Fresh Orders Data:")
orders_fresh.display()

# Stale orders (older than 24 hours)
stale_data = [
    (5, 105, 600, now - timedelta(hours=48)),
    (6, 106, 800, now - timedelta(hours=72)),
    (7, 107, 350, now - timedelta(days=5)),
    (8, 108, 450, now - timedelta(days=10))
]
orders_stale = spark.createDataFrame(stale_data, schema)

print("\n📊 Stale Orders Data:")
orders_stale.display()

# ============================================================================
# 5. SINGLE FRESHNESS CHECK
# ============================================================================
# Check freshness of the fresh orders table
print("\n🔍 Checking freshness of fresh orders (should PASS):")
result_fresh = check_freshness(orders_fresh, "order_date", max_age_hours=24)
print(f"Latest timestamp: {result_fresh['latest_timestamp']}")
print(f"Age in hours: {result_fresh['age_hours']}")
print(f"Passed: {result_fresh['passed']}")

# Check freshness of the stale orders table
print("\n🔍 Checking freshness of stale orders (should FAIL):")
result_stale = check_freshness(orders_stale, "order_date", max_age_hours=24)
print(f"Latest timestamp: {result_stale['latest_timestamp']}")
print(f"Age in hours: {result_stale['age_hours']}")
print(f"Passed: {result_stale['passed']}")

# ============================================================================
# 6. BATCH FRESHNESS CHECK
# ============================================================================
# Check multiple tables at once
table_registry = {
    "orders_fresh": orders_fresh,
    "orders_stale": orders_stale
}

freshness_config = [
    {"table": "orders_fresh", "timestamp_column": "order_date", "max_age_hours": 24},
    {"table": "orders_stale", "timestamp_column": "order_date", "max_age_hours": 24}
]

print("\n🔍 Running batch freshness checks:")
all_results = check_all_freshness(table_registry, freshness_config)

for result in all_results:
    status = "✅ PASSED" if result["passed"] else "❌ FAILED"
    print(f"\nTable: {result['table']}")
    print(f"Status: {status}")
    print(f"Latest timestamp: {result['latest_timestamp']}")
    print(f"Age: {result['age_hours']} hours")

In [0]:
# ============================================================================
# 7. SAVE RESULTS TO DELTA TABLE
# ============================================================================
# Convert the freshness check results to a DataFrame
results_df = spark.createDataFrame(all_results)

# Display the results DataFrame before saving
print("\n📋 Freshness Check Results Summary:")
results_df.display()

# Save results to Delta table for tracking and reporting
# Table: workspace.default.freshness_report
results_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.freshness_report")

print("\n✅ Results saved to 'workspace.default.freshness_report' table")

In [0]:
# ============================================================================
# 8. VERIFY SAVED RESULTS
# ============================================================================
# Read back the saved freshness report from Delta table to verify persistence
df = spark.table("workspace.default.freshness_report")

print("📊 Reading back from Delta table: workspace.default.freshness_report")
df.display()

In [0]:
# ============================================================================
# 9. BONUS: Get ALL Individual Stale Records
# ============================================================================
# The check_freshness() function tells us if a TABLE is fresh (being updated)
# But what if you want to see WHICH INDIVIDUAL RECORDS are stale?

from datetime import datetime, timedelta
from pyspark.sql.functions import col, current_timestamp, unix_timestamp

print("🔍 Finding ALL individual stale records (older than 24 hours):")
print("="*70)

# Calculate the cutoff time (24 hours ago)
max_age_hours = 24

# Filter for records older than the threshold
stale_records = orders_stale.filter(
    (unix_timestamp(current_timestamp()) - unix_timestamp(col("order_date"))) / 3600 > max_age_hours
)

print(f"\n📊 All stale records (older than {max_age_hours} hours):")
stale_records.display()

stale_count = stale_records.count()
print(f"\n📈 Total stale records: {stale_count}")
print(f"Expected: 4 records (orders 5, 6, 7, 8)")

# ============================================================================
# COMPARISON: Table-Level vs Record-Level Checks
# ============================================================================
print("\n" + "="*70)
print("📋 COMPARISON: Two Different Use Cases")
print("="*70)

print("\n1️⃣ TABLE-LEVEL FRESHNESS (check_freshness):")
print("   Question: 'Is this table being updated?'")
print("   Returns: ONE result per table")
print("   Use case: Pipeline monitoring, alerting on stale tables")
print("   Example: Alert if orders table hasn't received data in 24 hours")

print("\n2️⃣ RECORD-LEVEL FILTERING (SQL filter):")
print("   Question: 'Which individual records are old?'")
print("   Returns: ALL rows matching criteria")
print("   Use case: Data cleanup, archival, finding old transactions")
print("   Example: Archive orders older than 90 days")